In [5]:
# Set Project Root
import sys
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

%reload_ext autoreload
%autoreload 2

/teamspace/studios/this_studio/Airport-AI


In [2]:
# Verify datasets file
for folder in [
    "datasets/airport/train/images",
    "datasets/airport/train/labels",
    "datasets/airport/valid/images",
    "datasets/airport/valid/labels",
    "datasets/airport/test/images",
    "datasets/airport/test/labels",
]:
    p = PROJECT_ROOT / folder
    print(p)
    print("Exists:", p.exists())
    if p.exists():
        print("Files:", len(list(p.iterdir())))
    print()

/teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images
Exists: True
Files: 5676

/teamspace/studios/this_studio/Airport-AI/datasets/airport/train/labels
Exists: True
Files: 5676

/teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/images
Exists: True
Files: 811

/teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/labels
Exists: True
Files: 811

/teamspace/studios/this_studio/Airport-AI/datasets/airport/test/images
Exists: True
Files: 405

/teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels
Exists: True
Files: 405



In [3]:
# Verify the dataset
from ultralytics.data.utils import check_det_dataset

check_det_dataset(str(PROJECT_ROOT / "datasets/airport/data.yaml"))

print("Dataset verified successfully.")

Dataset verified successfully.


In [5]:
# Perform a 1-epoch smoke test
from ultralytics import YOLO

model = YOLO(PROJECT_ROOT / "models/yolo26n.pt")

model.train(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    device=0,
    cache=True,
)

New https://pypi.org/project/ultralytics/8.4.112 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/teamspace/studios/this_studi

 23        [16, 19, 22]  1    243516  ultralytics.nn.modules.head.Detect           [6, 1, True, [64, 128, 256]]  
YOLO26n summary: 260 layers, 2,506,140 parameters, 2,506,140 gradients, 5.8 GFLOPs

Transferred 606/708 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 720.2±213.5 MB/s, size: 55.3 KB)
train: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/labels... 5676 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5676/5676 1.0Kit/s 5.6s0.1ss
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/01_sample_000027_jpg.rf.d664ea3b99ace9605035f470dc26fb14.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/01_sample_000032_jpg.rf.20a27777446d6500d69936d3a6fe40d0.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79dfa02026f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

## Model Training Phase - 1

In [7]:
# Full Model training
model = YOLO(PROJECT_ROOT / "models/yolo26n.pt")

results = model.train(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",

    epochs=150,
    imgsz=640,

    batch=16,
    workers=8,
    device=0,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=5e-4,

    warmup_epochs=3,
    patience=25,

    cache=True,
    amp=True,

    pretrained=True,

    project=PROJECT_ROOT / "runs/train",
    name="airport_yolo26n",

    save=True,
    save_period=True,

    val=True,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.112 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/teamspace/studios/this_s

train: Caching images (6.5GB RAM): 100% ━━━━━━━━━━━━ 5676/5676 935.2it/s 6.1s<0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 57.4±70.6 MB/s, size: 54.2 KB)
val: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/labels.cache... 811 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 811/811 26.6Mit/s 0.0s
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/images/03_sample_000856_jpg.rf.a7c93582290dff5623e3db6dfab36308.jpg: 1 duplicate labels removed
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/valid/images/04_sample_000008_jpg.rf.1ce383715e57d657f93b48f75dcfe68c.jpg: 1 duplicate labels removed
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alte

In [8]:
# Model Evaluation
from ultralytics import YOLO

model = YOLO(PROJECT_ROOT / "runs/train/airport_yolo26n/weights/best.pt")

metrics = model.val(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",
    split="test",
    imgsz=640,
)

print(metrics)

Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 111.1±50.4 MB/s, size: 56.6 KB)
val: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels... 405 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 405/405 859.6it/s 0.5s0.1s
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/images/03_sample_000728_jpg.rf.ea536d3f6f5bea80f16a9901c5ac639e.jpg: 1 duplicate labels removed
val: New cache created: /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.0it/s 8.8s0.1s
                   all        405       3801      0.803       0.76      0.774      0.585
              aircraft        405        989      0.901      0.683       0.74      0.634
      

## Model Training Phase - 2

In [6]:
# Full Model training - Phase 2

model = YOLO(PROJECT_ROOT / "runs/train/airport_yolo26n/weights/best.pt")

results = model.train(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",

    epochs=250, # continue training
    imgsz=640, # unchanged

    batch=16,   # unchanged
    workers=8,  # unchanged
    device=0,   # unchanged

    optimizer="AdamW",  # unchanged
    lr0=0.0002, # 5x lower learning rate for fine-tuning
    lrf=0.001,
    weight_decay=5e-4,  # unchanged

    warmup_epochs=1,
    patience=50,    # allow further improvement

    cache=True, # unchanged
    amp=True,   # unchanged

    pretrained=False,   # already loading pretrained weights

    project=PROJECT_ROOT / "runs/train",
    name="airport_yolo26n_phase2",

    save=True,
    save_period=True,

    val=True,
    plots=True
)

New https://pypi.org/project/ultralytics/8.4.113 available 😃 Update with 'pip install -U ultralytics'


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=250, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/teamspace/studios/this_studio/Airport-AI/runs/train/airport_yolo26n/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale

AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1088.5±477.0 MB/s, size: 58.8 KB)
train: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/labels.cache... 5676 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5676/5676 850.2Mit/s 0.0s
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/01_sample_000027_jpg.rf.d664ea3b99ace9605035f470dc26fb14.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/01_sample_000032_jpg.rf.20a27777446d6500d69936d3a6fe40d0.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/03_sample_001778_jpg.rf.3d8872febe9c4c03786678bd932ce8c8.jpg: 1 duplicate labels removed
train: /teamspace/studios/this_studio/Airport-AI/datasets/airport/train/images/03_sample_001779_jpg.rf.64f5d49a2418952b39cc2dee068ee4cc.jpg: 1 duplicate l

In [7]:
# Model Evaluation
from ultralytics import YOLO

model = YOLO(PROJECT_ROOT / "runs/train/airport_yolo26n_phase2/weights/best.pt")

metrics = model.val(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",
    split="test",
    imgsz=640,
)

print(metrics)

Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 106.6±39.4 MB/s, size: 51.1 KB)
val: Scanning /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/labels.cache... 405 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 405/405 67.9Mit/s 0.0s
val: /teamspace/studios/this_studio/Airport-AI/datasets/airport/test/images/03_sample_000728_jpg.rf.ea536d3f6f5bea80f16a9901c5ac639e.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.5it/s 7.5s0.1s
                   all        405       3801      0.814      0.687      0.734       0.52
              aircraft        405        989      0.914      0.626      0.698       0.59
           airport_tug         27         69      0.655      0.188       0.27      0.155
                

## Model Training Phase 3

In [8]:
from ultralytics import YOLO

model = YOLO(
    PROJECT_ROOT /
    "runs/train/airport_yolo26n_phase2/weights/best.pt"
)

results = model.train(
    data=PROJECT_ROOT / "datasets/airport/data.yaml",
    epochs=100,
    imgsz=768,
    batch=16,
    workers=8,
    device=0,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=1e-5,
    weight_decay=5e-4,
    warmup_epochs=0,
    patience=30,
    cache=True,
    amp=True,
    pretrained=False,
    project=PROJECT_ROOT / "runs/train",
    name="airport_yolo26n_phase3",
    save=True,
    save_period=10,
    val=True,
    plots=True,
)

New https://pypi.org/project/ultralytics/8.4.113 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.11 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/teamspace/studios/this_studio/Airport-AI/datasets/airport/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=1e-05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/teamspace/studios/this

## Best Model Export

In [ ]:
# Export
# from ultralytics import YOLO

# model = YOLO(PROJECT_ROOT / "runs/train/airport_yolo26n/weights/best.pt")

# model.export(format="onnx")